### ailen vs predator 데이터셋
- https://www.kaggle.com/datasets/pmigdal/alien-vs-predator-images?resource=download

### Alexnet 활용

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

In [ ]:
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomAffine(0, shear=10, scale=(0.8, 1.2)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor()
    ]),
    'validation': transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor()
    ])
}

In [ ]:
def target_transforms(target):
    return torch.FloatTensor([target])

In [ ]:
image_datasets = {
    'train' : datasets.ImageFolder('./dataset/AilenPredator/data/train', data_transforms['train'], target_transform=target_transforms),
    'validation': datasets.ImageFolder('./dataset/AilenPredator/data/validation', data_transforms['validation'], target_transform = target_transforms)
}

In [ ]:
dataloaders = {
    'train': DataLoader(
        image_datasets['train'],
        batch_size = 32,
        shuffle = True
    ),
    'validation': DataLoader(
        image_datasets['validation'],
        batch_size = 32,
        shuffle = False
    )
}

In [ ]:
len(image_datasets['train']),len(image_datasets['validation'])
# 694,200

In [ ]:
imgs, labels = next(iter(dataloaders['train']))

fig, axes = plt.subplots(4,8, figsize = (16,8))

for ax, img, label in zip(axes.flatten(), imgs, labels):
    ax.imshow(img.permute(1,2,0))
    ax.set_title(label.item())
    ax.axis('off')

In [ ]:
model = models.alexnet(weights = 'IMAGENET1K_V1').to(device)
print(model)

In [ ]:
for param in model.parameters():
    param.requires_glad = False

In [ ]:
model.classifier = nn.Sequential(
    nn.Linear(256 * 6 * 6, 128),
    nn.ReLU(),
    nn.Linear(128, 1), # 여기선 1로 빼서 확률으로 분류해줌
    nn.Sigmoid()
).to(device)

model

In [ ]:
optimizer = optim.Adam(model.classifier.parameters(), lr= 0.001)# 모델의 classifier의 파라미터만 수정함

In [ ]:
epochs = 10

for epoch in range(epochs):
    for phase in ['train', 'validation']:
        if phase == 'train':
            model.train()
        else:
            model.eval()

        sum_losses = 0
        sum_accs = 0

        for x_batch, y_batch in dataloaders[phase]:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            
            y_pred = model(x_batch)
            loss = nn.BCELoss()(y_pred, y_batch)

            if phase == 'train':
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            sum_losses = sum_losses + loss
            y_bool = (y_pred >= 0.5).float()
            acc = (y_batch == y_bool).float().sum()/len(y_batch) * 100
            sum_accs = sum_accs + acc

        avg_loss = sum_losses / len(dataloaders[phase])
        avg_acc = sum_accs / len(dataloaders[phase])
        print(f'{phase:10s}: Epoch {epoch+1:4d}/{epochs} Loss: {avg_loss: .4f} Accuracy: {avg_acc:.2f}%')

In [ ]:
from PIL import Image